# Práctica de RAG sobre la Ley 21.526 — caso de uso real (legal)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_legal.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** aplicar el pipeline RAG con LangChain (`EnsembleRetriever` con RRF + cross-encoder reranker) sobre un corpus **real, extenso y técnico**: la Ley argentina 21.526 de Entidades Financieras (~91k chars, 67 artículos).
>
> **¿Por qué esta notebook?** En `practica_completa.ipynb` vimos que sobre un corpus chico (programa de cátedra, 5 docs) el reranker **no daba ganancia clara** sobre hybrid+RRF. Acá probamos sobre un corpus realista (legal, terminología técnica) y vemos cuándo el reranker **sí** agrega valor.
>
> **Estructura:**
> 1. Setup — instalar deps, configurar GROQ.
> 2. Descargar y parsear la Ley 21.526 desde una URL pública.
> 3. Chunking por artículo (regex que respeta el formato legal).
> 4. Construir retrievers: dense, BM25, hybrid (RRF), hybrid + reranker.
> 5. **Comparación cualitativa de 3 casos** con análisis del por qué de cada resultado.
> 6. Vista panorámica con benchmark sobre 15 queries.
> 7. Cierre — qué te llevás.
>
> **Tiempo:** ~25 min en Colab (incluyendo descarga del reranker, ~280 MB).

## 0. Setup

In [ ]:
!pip install -q sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib pypdf requests langchain langchain-community langchain-classic langchain-chroma langchain-huggingface


In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')

In [ ]:
import os

# En Colab podés usar Secrets ("llave" en la barra lateral)
try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')

In [ ]:
from groq import Groq

_groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])


def llamar_llm(messages, model='llama-3.1-8b-instant', temperature=0.1, max_tokens=1024):
    """Wrapper de Groq. Temperature baja para reproducibilidad pedagógica."""
    resp = _groq_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


print(llamar_llm([{'role': 'user', 'content': 'Decime "ok" si me podés escuchar'}]))

## 1. Descargar y parsear la Ley 21.526

Bajamos el PDF desde la web pública de la OEA y extraemos el texto con `pypdf`. La ley regula a las entidades financieras en Argentina — bancos comerciales, de inversión, hipotecarios, compañías financieras, cajas de crédito, etc. Tiene 67 artículos agrupados en Títulos y Capítulos.

**¿Por qué este corpus para una clase de RAG?**

- **Tamaño realista** — ~91k chars vs los ~6k del programa de cátedra. El BM25 tiene más vocabulario, el dense tiene más vecinos semánticos, hay margen real para que las técnicas de retrieval se distingan.
- **Terminología técnica** — el lenguaje legal usa palabras como "revocación", "intermediación habitual", "obligación de guardar secreto" que NO son las que un usuario común usaría ("¿cómo cancelan un banco?", "¿pueden contar mis movimientos?"). Esto fuerza al retrieval a hacer trabajo semántico real.
- **Documento público y citable** — no hay problemas de privacidad y cualquiera puede verificar las respuestas contra el texto original.

In [ ]:
import os
import requests

PDF_PATH = 'ley21526.pdf'
# Fuente original (puede 403/timeout): https://www.oas.org/juridico/PDFs/mesicic3_arg_ley21526.pdf
# Mirrors confiables en este mismo repo — probamos main primero, después la branch de desarrollo.
URLS = [
    'https://raw.githubusercontent.com/jorgeroa/ia-utn-frsf/main/clase03/notebooks/rag/ley21526.pdf',
    'https://raw.githubusercontent.com/jorgeroa/ia-utn-frsf/feat/clase03/clase03/notebooks/rag/ley21526.pdf',
]

if not os.path.exists(PDF_PATH):
    last_error = None
    for url in URLS:
        try:
            print(f'Descargando PDF desde {url}...')
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(PDF_PATH, 'wb') as f:
                f.write(r.content)
            print(f'✓ Descargado: {os.path.getsize(PDF_PATH)} bytes')
            break
        except requests.HTTPError as e:
            print(f'  (404 en esta URL, probando la siguiente...)')
            last_error = e
    else:
        raise RuntimeError(f'No se pudo descargar el PDF desde ningún mirror: {last_error}')
else:
    print(f'✓ PDF local encontrado: {os.path.getsize(PDF_PATH)} bytes')

from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
full_text = "\n".join(page.extract_text() for page in reader.pages)
print(f'✓ {len(reader.pages)} páginas, {len(full_text)} caracteres extraídos')
print('\nPrimeras líneas:')
print(full_text[:400])


## 2. Chunking por artículo

Los textos legales tienen una estructura natural: cada artículo es una unidad lógica autocontenida. Chunkeamos por artículo en lugar de por oración o tamaño fijo.

**Trampa visible:** algunos artículos tienen un footnote marker entre el número y el guión:

```
Art. 4 (1) – Autoridad. Funciones*.
Art. 35 (2) – Régimen sancionatorio.
```

Si el regex no contempla el `(1)`, el artículo se "pega" al chunk anterior y se pierde. Esto pasó en una versión previa de esta misma práctica — la query *"¿qué autoridad regula?"* devolvía el Art 3 en lugar del Art 4 porque el contenido del 4 estaba embebido dentro del 3. Es **un error sutil que sólo se ve cuando comparás retrieval contra la verdad del corpus**.

> Moraleja pre-RAG: tu chunking sólo es bueno hasta donde testeás. Mirá el corpus, no asumas que tu regex anda.

In [ ]:
import re

# Regex: split antes de "Art. N -" o "Art. N (footnote) -"
parts = re.split(r'(?=Art\.\s*\d+(?:\s*\([^)]+\))?\s*[\u2013\u2014\-])', full_text)
parts = [p.strip() for p in parts if p.strip()]

from langchain_core.documents import Document

docs = []
for p in parts:
    m = re.match(r'Art\.\s*(\d+)(?:\s*\([^)]+\))?\s*[\u2013\u2014\-]\s*([^\.\n]+?)\.?', p)
    if m:
        art_num = int(m.group(1))
        titulo = m.group(2).strip()[:60]
        body = p[:1800]  # cap article body length
        docs.append(Document(
            page_content=body,
            metadata={'art': art_num, 'titulo': titulo},
        ))

print(f'✓ {len(docs)} artículos chunkeados')
art_nums = sorted({d.metadata['art'] for d in docs})
print(f'  Rango: Art {art_nums[0]} - Art {art_nums[-1]}')
print(f'  Total únicos: {len(art_nums)}')
print()
print('Primeros 5 artículos:')
for d in docs[:5]:
    print(f'  Art {d.metadata["art"]:2}: {d.metadata["titulo"]}')

## 3. Embeddings + ChromaDB

Usamos el **mismo embedding** que en `practica_completa.ipynb`: `paraphrase-multilingual-MiniLM-L12-v2`. Es chico (~85 MB), multilingual, y suficiente — el embedding mejor (ej. e5-large) no aporta mejoras dramáticas con un chunking correcto, y agrega 2 GB de descarga.

> **Decisión consciente**: en producción real para legal-es probablemente querrías un embedding fine-tuned o más grande. Para esta clase priorizamos baja fricción de setup.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

emb = HuggingFaceEmbeddings(model_name='paraphrase-multilingual-MiniLM-L12-v2')
vs = Chroma.from_documents(docs, emb, collection_name='ley21526')
print(f'✓ {vs._collection.count()} chunks indexados en ChromaDB')

## 4. Retrievers: dense, BM25, hybrid (RRF), hybrid + reranker

Construimos los 4 retrievers para poder compararlos:

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Dense retriever (sólo embeddings)
dense = vs.as_retriever(search_kwargs={'k': 3})

# BM25 (sólo léxico)
bm25 = BM25Retriever.from_documents(docs)
bm25.k = 3

# Hybrid con RRF (BM25 + dense, fusión por ranks)
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])

# Para reranking, subimos k a 10 en cada uno (más candidatos para el cross-encoder)
bm25_rk = BM25Retriever.from_documents(docs); bm25_rk.k = 10
dense_rk = vs.as_retriever(search_kwargs={'k': 10})
hybrid_rk = EnsembleRetriever(retrievers=[bm25_rk, dense_rk], weights=[0.5, 0.5])

ce = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=ce, top_n=3)
hybrid_rerank = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=hybrid_rk
)

print('✓ Retrievers listos: dense, bm25, hybrid, hybrid_rerank')

In [ ]:
SYSTEM_LEGAL = (
    'Sos un asistente legal experto en la Ley argentina 21.526 de Entidades Financieras. '
    'Respondé la pregunta usando ÚNICAMENTE el contexto provisto. '
    'IMPORTANTE: leé TODOS los artículos del contexto e integrá la información relevante de cada uno. '
    'Citá explícitamente el número de artículo cuando corresponda. '
    'Sé concreto, citá el texto, máximo 6 oraciones. '
    'Si el contexto no alcanza, decilo brevemente.'
)


def rag_legal(query, retriever):
    """RAG query sobre Ley 21.526 — devuelve respuesta + artículos retrieved."""
    ds = retriever.invoke(query)
    if len(ds) > 6:
        ds = ds[:6]
    arts = [d.metadata['art'] for d in ds]
    contexto = '\n\n'.join(f'[Art {d.metadata["art"]}] {d.page_content}' for d in ds)
    resp = llamar_llm([
        {'role': 'system', 'content': SYSTEM_LEGAL},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {query}'},
    ])
    return resp, arts


print('✓ rag_legal() listo')

## 5. Dos casos — naive vs hybrid

Comparamos **dos pipelines** sobre el mismo corpus:

| Pipeline | Componentes |
|---|---|
| **Naive** | dense retrieval (embeddings + ChromaDB) — el RAG mínimo. |
| **Hybrid** | BM25 + dense fusionados con RRF (`EnsembleRetriever`). |

Y dos queries:

| Caso | Query | Lo que mostramos |
|---|---|---|
| **1 — Dense alcanza** | Pregunta con vocabulario que se cruza con el corpus. | Naive responde bien. **No necesitás complejidad.** |
| **2 — Hybrid rescata** | Pregunta con jerga técnica específica. | Naive se desvía a artículos cercanos pero incorrectos; hybrid trae el artículo correcto gracias a BM25. |

**¿Y el reranker?** Lo construimos en la Sección 4 pero **no lo usamos como demo "win" acá**. ¿Por qué? Sobre este corpus (67 artículos), el cross-encoder muestra **valor estadístico, no determinista** — algunas queries mejoran, otras empeoran. La ventaja del rerank emerge claramente en **corpus mucho más grandes**. En la Sección 6 te dejamos ejercicios para explorarlo en tu caso.

> **¿Por qué cualitativo y no aggregate score?** La métrica de keywords cuenta presencia literal — está mal calibrada para comparar técnicas. La evaluación seria de un sistema RAG (LLM-as-judge, RAGAS, anotación humana) la ves en **Clase 3b**.


In [ ]:
def comparar(titulo, query, expected_arts, retrievers):
    """Corre 2+ retrievers para una query e imprime side-by-side.

    `retrievers` es una lista [(label, retriever), ...]."""
    print('═' * 95)
    print(f'  {titulo}')
    print('═' * 95)
    print(f'Query:    "{query}"')
    print(f'Esperado: artículos {expected_arts}')
    print()

    for label, ret in retrievers:
        resp, arts = rag_legal(query, ret)
        hits = [a for a in arts if a in expected_arts]
        print(f'── {label} ──')
        print(f'   sources: {arts}    esperados encontrados: {hits}')
        print(f'   resp:')
        for line in resp.split('\n'):
            print(f'     {line}')
        print()


print('✓ comparar() listo')


### Caso 1 — Naive funciona

**Query:** *"¿Qué operaciones puede realizar un banco comercial?"*

**Respuesta correcta** está en el Art 21 (que enumera las operaciones permitidas a bancos comerciales).

**Predicción:** la query usa vocabulario que se cruza con el título y el cuerpo del Art 21 ("banco comercial", "operaciones"). El **dense retriever solo** (sin BM25, sin reranker, sin hybrid) debería encontrar el artículo correcto sin esfuerzo.

**Lección:** no necesitás complejidad cuando la query y el corpus comparten vocabulario natural.


In [ ]:
comparar(
    'CASO 1 — Naive funciona: ¿qué operaciones puede realizar un banco comercial?',
    '¿Qué operaciones puede realizar un banco comercial?',
    expected_arts=[21],
    retrievers=[('rag naive (dense only)', dense)],
)

print('💡 Qué mirar:')
print('   La respuesta debe citar el Art 21 explícitamente y enumerar varias operaciones')
print('   típicas: recibir depósitos, otorgar créditos, comprar y vender títulos, etc.')
print('   Conclusión: cuando la query encaja léxica y semánticamente con el corpus,')
print('   dense solo basta. Avanzar a hybrid/rerank sería sobre-ingeniería.')


### Caso 2 — Hybrid rescata

**Query:** *"¿Qué pasa si una entidad financiera tiene problemas de iliquidez?"*

**Respuesta correcta** está en el Art 35 ("Iliquidez transitoria. Auxilio financiero").

**Predicción:** "iliquidez" es un **término técnico específico** que aparece sólo en el Art 35. El dense retriever, al generalizar semánticamente, asocia "problemas de iliquidez" con conceptos amplios como *"liquidación"* (Art 49-50) o *"saneamiento"* (Art 31) — y trae chunks parecidos pero **incorrectos**.

**BM25** busca el token "iliquidez" exacto y lo encuentra en el Art 35. El hybrid (BM25 + dense + RRF) fusiona ambas señales y **rescata el Art 35** al top-3, **de forma determinista** (BM25 no depende de seeds aleatorios).

**Lección:** cuando tu query tiene **jerga específica** (legal, médica, técnica, siglas, IDs), el matching léxico de BM25 es irreemplazable. Ese es el caso donde hybrid agrega valor sobre dense puro.


In [ ]:
comparar(
    'CASO 2 — Hybrid rescata: ¿qué pasa con iliquidez?',
    '¿Qué pasa si una entidad financiera tiene problemas de iliquidez?',
    expected_arts=[35],
    retrievers=[
        ('rag naive (dense only)', dense),
        ('rag hybrid (BM25 + dense + RRF)', hybrid),
    ],
)

print('💡 Qué mirar:')
print('   La respuesta de naive probablemente menciona "liquidación" (Arts 49-50) o')
print('   "saneamiento" (Art 31) — temas tangencialmente relacionados pero NO la respuesta.')
print('   La respuesta de hybrid debería citar el Art 35 ("Iliquidez transitoria") y')
print('   mencionar el "auxilio financiero" del Banco Central.')
print()
print('   La diferencia salió porque BM25 captó el token literal "iliquidez". Sin BM25,')
print('   dense solo se quedó en vecinos semánticos vagos.')


## 6. Tu turno — exploración guiada

Los 3 casos anteriores muestran un patrón claro pero **cherry-picked** — elegimos las queries para ilustrar el escalado. En la realidad **vas a tener queries de los 3 tipos mezcladas** y es ahí donde hay que medir.

Te dejamos un set de 12 queries variadas y dos retrievers (`hybrid` y `hybrid_rerank`) para que exploremos juntos qué pasa en cada una. **No corremos el LLM** — sólo retrieval, así podés iterar rápido y mirar muchas queries.

Después del benchmark, hay sugerencias de variantes para que pruebes vos.


In [ ]:
QUERIES = [
    ('q1',  '¿Qué autoridad regula las entidades financieras y qué facultades tiene?', [4]),
    ('q2',  '¿Qué requisitos debe cumplir una persona para ser director o síndico?', [10]),
    ('q3',  '¿Cuáles son las operaciones permitidas a un banco de inversión?', [22]),
    ('q4',  '¿Qué pasa si una entidad financiera viola el secreto bancario?', [39, 40, 41]),
    ('q5',  '¿Cuál es el procedimiento para revocar la autorización para funcionar?', [44, 45]),
    ('q6',  '¿Qué obligaciones de información tienen las entidades hacia el Banco Central?', [36, 37]),
    ('q7',  '¿Cuándo procede la liquidación de una entidad financiera?', [49, 50]),
    ('q8',  '¿Qué garantías existen sobre los depósitos bancarios?', [56]),
    ('q9',  '¿Qué normas se aplican a las cajas de crédito?', [26]),
    ('q10', '¿Cómo se aplican las sanciones a una entidad infractora?', [41, 42]),
    ('q11', '¿Qué pasa con los depositantes en caso de quiebra de una entidad?', [53, 54, 55]),
    ('q12', '¿Cómo se realiza el control de las operaciones por el Banco Central?', [37, 38]),
]

import pandas as pd

rows = []
for qid, query, expected in QUERIES:
    d_arts = [d.metadata['art'] for d in dense.invoke(query)]
    h_arts = [d.metadata['art'] for d in hybrid.invoke(query)[:3]]
    r_arts = [d.metadata['art'] for d in hybrid_rerank.invoke(query)]
    d_hits = len([a for a in d_arts if a in expected])
    h_hits = len([a for a in h_arts if a in expected])
    r_hits = len([a for a in r_arts if a in expected])

    rows.append({
        'id': qid,
        'query': query[:55] + ('...' if len(query) > 55 else ''),
        'esperado': str(expected),
        'dense': f'{d_hits}/{len(expected)}  {d_arts}',
        'hybrid': f'{h_hits}/{len(expected)}  {h_arts}',
        'rerank': f'{r_hits}/{len(expected)}  {r_arts}',
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)
print(df.to_string(index=False))
print()

# Resumen agregado
total_d = sum(min(1, len([a for a in [d.metadata['art'] for d in dense.invoke(q)] if a in exp])) for _, q, exp in QUERIES)
total_h = sum(min(1, len([a for a in [d.metadata['art'] for d in hybrid.invoke(q)[:3]] if a in exp])) for _, q, exp in QUERIES)
total_r = sum(min(1, len([a for a in [d.metadata['art'] for d in hybrid_rerank.invoke(q)] if a in exp])) for _, q, exp in QUERIES)

print(f'Queries con ≥1 artículo esperado en top-3:')
print(f'  dense:  {total_d}/{len(QUERIES)}')
print(f'  hybrid: {total_h}/{len(QUERIES)}')
print(f'  rerank: {total_r}/{len(QUERIES)}')
print()
print('💡 Mirá la tabla query por query. Lo importante NO es el total sino:')
print('   - qué tipo de queries cada técnica resuelve mejor')
print('   - dónde TODAS fallan (problema de embedding o vocabulario)')
print('   - dónde más vale un cambio simple (mejor embedding, query rewriting)')
print('     que sumar más capas al pipeline.')


### Sugerencias de exploración

Ahora **vos** probás. Sugerencias:

1. **Agregá tus propias queries** — pensá una pregunta legal que harías como ciudadano y testeala contra los 3 retrievers. ¿Encuentra el artículo correcto?
2. **Variá `n_chunks`** — re-construí `dense = vs.as_retriever(search_kwargs={'k': 1})`. Con sólo 1 chunk al LLM, la **precisión en top-1** importa muchísimo. ¿El rerank empieza a brillar más?
3. **Variá los pesos del hybrid** — `EnsembleRetriever(retrievers=[bm25, dense], weights=[0.7, 0.3])` favorece BM25. Probá `[0.3, 0.7]` (favorece dense). ¿Cambia el ganador en queries específicas?
4. **Cambiá el embedding** — probá `BAAI/bge-m3` o `intfloat/multilingual-e5-large` (los dos pesan ~2 GB, tardan más en cargar). ¿Las queries que fallaban antes empiezan a funcionar?
5. **Probá `top_n=5` en el reranker** — `CrossEncoderReranker(model=ce, top_n=5)`. Con más chunks al LLM, ¿cambia algo? ¿Se vuelve más completo o ruidoso?
6. **Cambiá el reranker** — `jinaai/jina-reranker-v2-base-multilingual` o `BAAI/bge-reranker-v2-m3`. ¿Algún reranker es notablemente mejor en español jurídico?

> Cada uno de esos cambios es una **decisión real de ingeniería** que vas a tener que tomar en un proyecto de RAG. Sin medir, son decisiones a ciegas.


## 7. Cierre — qué te llevás

### Naive vs Hybrid

Los dos casos mostraron el patrón:

- **Cuando la query y el corpus comparten vocabulario natural**, dense solo alcanza. No necesitás complejidad.
- **Cuando la query tiene jerga técnica** (legal, médica, IDs, siglas), dense se desvía y **hybrid rescata** gracias al matching léxico exacto de BM25. RRF fusiona sin que las técnicas se peleen.

### La regla de oro

> **Empezá por dense solo.** Si funciona en TU corpus con TUS queries, **dejalo así**. Subí a hybrid cuando veas que tu dominio tiene términos técnicos que el embedding no captura. Subí al reranker (o HyDE, o parent-child) **sólo cuando una métrica concreta** te diga que vale la pena.

### Sobre el reranker (y por qué no lo demostramos como "win" acá)

El cross-encoder (`BAAI/bge-reranker-base`) está construido en la Sección 4. Lo podés usar en la Sección 6 (exploración). Pero **en este corpus de 67 artículos no encontramos una query donde el reranker gane *cleanly* sobre hybrid solo**. Más aún: en algunas corridas el rerank **empeoraba** al filtrar de más.

Eso no significa que el reranker no sirva. Significa que:
- Su valor es **estadístico, no determinista** — varía por query, por corpus, por seed.
- Su valor emerge claramente en corpus **muy grandes** (miles de chunks donde el top-k del bi-encoder está realmente ruidoso).
- En corpus chico-medio, **medilo en TU caso** antes de pagar la latencia y los ~280 MB del modelo.

Esto es **exactamente lo que vas a ver en producción**, y por eso la última palabra siempre la tiene la **medición end-to-end**.

### Lo que NO te enseñamos acá pero existe

- **HyDE** (Hypothetical Document Embeddings) — generar un documento sintético antes del embedding.
- **Parent-child chunking** — chunks chicos para retrieval, chunks grandes para el LLM.
- **Multi-query retrieval** — generar variantes de la query y unir resultados.
- **Embeddings fine-tuneados por dominio** — la palanca más fuerte cuando ninguna técnica genérica cierra (especialmente en español jurídico/médico).

### El paso obligado

Cualquier decisión de *"¿qué pipeline uso?"* debería estar **respaldada por métricas**, no por intuición o por hype. Eso es **Clase 3b**: cómo medir un sistema RAG con LLM-as-judge, RAGAS, observability con Arize AX, y red teaming para los casos donde el LLM "miente con confianza".
